# GMM Noise Generation on Robust Dataset

This notebook applies GMM-based noise augmentation to the robust-scaled ILPD dataset and writes a new augmented dataset.

In [ ]:
from pathlib import Path
import json
import tomllib
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture


In [ ]:
def resolve_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() or (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root from current working directory")

ROOT = resolve_repo_root(Path.cwd())
CONFIG_PATH = ROOT / "config" / "noise_generation.toml"
INPUT_PATH = ROOT / "data" / "processed" / "ILPD_robust_scaled_features.csv"
OUTPUT_PATH = ROOT / "data" / "processed" / "ILPD_robust_scaled_with_gmm_noise.csv"
REPORT_PATH = ROOT / "produced_reports" / "docs" / "ILPD_noise_generation_summary.json"

REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f"Config: {CONFIG_PATH}")
print(f"Input: {INPUT_PATH}")
print(f"Output dataset: {OUTPUT_PATH}")
print(f"Summary report: {REPORT_PATH}")

## 1) Load robust dataset

In [ ]:
df = pd.read_csv(INPUT_PATH)
print(df.shape)
df.head()

## 2) Legacy workflow extraction (GMM noise generation)

Extracted core logic from legacy implementation:
1. Fit GMM on feature subspaces (legacy did this by gender for healthy-class augmentation).
2. Sample synthetic candidates from the fitted GMM.
3. Add small Gaussian jitter to avoid mode collapse around component means.
4. Enforce structural constraints (e.g., binary gender encoding).
5. Apply clipping/filtering constraints before acceptance.
6. Merge accepted synthetic rows with real rows and save augmented dataset.

## 3) Configure and generate GMM noise rows on robust-scaled data

In [ ]:
with open(CONFIG_PATH, "rb") as f:
    cfg = tomllib.load(f)

NOISE_RATIO = float(cfg["noise"]["noise_ratio"])
N_COMPONENTS = int(cfg["gmm"]["n_components"])
JITTER_SCALE = float(cfg["gmm"]["jitter_scale"])
RANDOM_SEED = int(cfg["gmm"]["random_seed"])
GROUP_COLS = ["Result", "Gender"]  # preserve label and gender distribution

print("Loaded config:", cfg)

rng = np.random.default_rng(RANDOM_SEED)

feature_cols = [c for c in df.columns if c != "Result"]
numeric_feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df[c])]

synthetic_parts = []
summary_rows = []

for group_key, g in df.groupby(GROUP_COLS, dropna=False):
    g = g.reset_index(drop=True)
    n_real = len(g)
    n_target = max(1, int(np.round(n_real * NOISE_RATIO)))

    X_real = g[numeric_feature_cols].values.astype(float)

    # Fit GMM on group feature distribution
    n_comp = max(1, min(N_COMPONENTS, n_real - 1)) if n_real > 1 else 1
    gmm = GaussianMixture(
        n_components=n_comp,
        covariance_type="full",
        random_state=int(rng.integers(0, 2**31)),
        max_iter=500,
        n_init=3,
    )

    if n_real > 1:
        gmm.fit(X_real)
        samples, _ = gmm.sample(n_target)
    else:
        samples = np.repeat(X_real, n_target, axis=0)

    # Add small jitter
    stds = X_real.std(axis=0)
    jitter = rng.normal(loc=0.0, scale=JITTER_SCALE * np.where(stds == 0, 1.0, stds), size=samples.shape)
    samples = samples + jitter

    # Clip sampled features to observed robust range for stability
    mins = X_real.min(axis=0)
    maxs = X_real.max(axis=0)
    samples = np.clip(samples, mins, maxs)

    synth = pd.DataFrame(samples, columns=numeric_feature_cols)

    # Re-attach grouping columns (Result + Gender)
    if isinstance(group_key, tuple):
        for col, val in zip(GROUP_COLS, group_key):
            synth[col] = val
    else:
        synth[GROUP_COLS[0]] = group_key

    # Enforce binary Gender after noise path
    if "Gender" in synth.columns:
        synth["Gender"] = synth["Gender"].round().clip(0, 1)

    # Ensure all original columns exist in same order
    for col in df.columns:
        if col not in synth.columns:
            synth[col] = g.iloc[0][col]
    synth = synth[df.columns]
    synth["Synthetic_Flag"] = 1

    synthetic_parts.append(synth)
    summary_rows.append({
        "group": str(group_key),
        "n_real": int(n_real),
        "n_synthetic": int(len(synth)),
        "gmm_components_used": int(n_comp),
    })

original = df.copy()
original["Synthetic_Flag"] = 0
synthetic_df = pd.concat(synthetic_parts, ignore_index=True) if synthetic_parts else pd.DataFrame(columns=original.columns)
augmented_df = pd.concat([original, synthetic_df], ignore_index=True)

print("Original rows:", len(original))
print("Synthetic rows:", len(synthetic_df))
print("Augmented rows:", len(augmented_df))
augmented_df.head()

## 4) Save augmented dataset + generation summary

In [ ]:
augmented_df.to_csv(OUTPUT_PATH, index=False)

summary = {
    "input_path": str(INPUT_PATH),
    "output_path": str(OUTPUT_PATH),
    "noise_ratio": NOISE_RATIO,
    "n_components_requested": N_COMPONENTS,
    "jitter_scale": JITTER_SCALE,
    "random_seed": RANDOM_SEED,
    "rows_original": int(len(original)),
    "rows_synthetic": int(len(synthetic_df)),
    "rows_total": int(len(augmented_df)),
    "group_summary": summary_rows,
}

REPORT_PATH.write_text(json.dumps(summary, indent=2))

print("Saved:")
print(OUTPUT_PATH)
print(REPORT_PATH)

## Export notebook to HTML report

In [ ]:
import subprocess

HTML_DIR = ROOT / "produced_reports"
HTML_DIR.mkdir(parents=True, exist_ok=True)

NOTEBOOK_PATH = ROOT / "scripts" / "04_noise_generation_gmm.ipynb"

subprocess.run([
    "jupyter", "nbconvert",
    "--to", "html",
    str(NOTEBOOK_PATH),
    "--output-dir", str(HTML_DIR),
], check=True)

print(f"Exported HTML to: {HTML_DIR / NOTEBOOK_PATH.with_suffix('.html').name}")